In [1]:
# GPU Check and System Information
import torch
import subprocess
import psutil

print("="*80)
print("🖥️  SYSTEM INFORMATION & GPU CHECK")
print("="*80)

# Check PyTorch and CUDA
print(f"\n📦 PyTorch Version: {torch.__version__}")
print(f"🔧 CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n🎮 GPU Information:")
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"   Free Memory: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB allocated")
    print(f"   CUDA Version: {torch.version.cuda}")
    
    # Run nvidia-smi
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,utilization.gpu', 
                               '--format=csv,noheader,nounits'], 
                              capture_output=True, text=True)
        gpu_info = result.stdout.strip().split(', ')
        print(f"\n📊 nvidia-smi:")
        print(f"   GPU: {gpu_info[0]}")
        print(f"   Total VRAM: {float(gpu_info[1])/1024:.1f} GB")
        print(f"   Free VRAM: {float(gpu_info[2])/1024:.1f} GB")
        print(f"   GPU Utilization: {gpu_info[3]}%")
    except:
        print("   (nvidia-smi not available)")
else:
    print("\n❌ CUDA NOT AVAILABLE - This benchmark requires a GPU!")
    print("   Please check your PyTorch installation and GPU drivers.")

# System info
print(f"\n💻 System Resources:")
print(f"   CPU Cores: {psutil.cpu_count()}")
print(f"   RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"   Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")

print("\n" + "="*80)
if torch.cuda.is_available():
    print("✅ GPU CHECK PASSED - Ready to start benchmark!")
else:
    print("❌ GPU CHECK FAILED - Cannot proceed without GPU")
print("="*80)

🖥️  SYSTEM INFORMATION & GPU CHECK

📦 PyTorch Version: 2.8.0+cu128
🔧 CUDA Available: True

🎮 GPU Information:
   Device: NVIDIA GeForce RTX 4090
   Total Memory: 23.6 GB
   Free Memory: 0.0 GB allocated
   CUDA Version: 12.8

📊 nvidia-smi:
   GPU: NVIDIA GeForce RTX 4090
   Total VRAM: 24.0 GB
   Free VRAM: 2.4 GB
   GPU Utilization: 0%

💻 System Resources:
   CPU Cores: 32
   RAM: 62.6 GB
   Available RAM: 9.8 GB

✅ GPU CHECK PASSED - Ready to start benchmark!


# Extensive Multi-Dataset Benchmark: Qwen2.5-7B vs Qwen3-30B

## 🎯 Comprehensive PhD-Level Evaluation

### Datasets Tested:
1. **Android** - Mobile application logs
2. **Apache** - Web server logs
3. **BGL** - Blue Gene/L supercomputer logs
4. **Hadoop** - Distributed computing framework logs
5. **HDFS** - Hadoop distributed file system logs
6. **HealthApp** - Health application logs
7. **HPC** - High-performance computing logs
8. **OpenStack** - Cloud infrastructure logs
9. **Spark** - Big data processing logs
10. **Thunderbird** - Supercomputer logs
11. **Zookeeper** - Coordination service logs

### Metrics Evaluated:
- **PA (Parsing Accuracy)**: Correct template matching %
- **GA (Grouping Accuracy)**: Clustering quality (Fixed algorithm)
- **Generation Time**: Parser bank creation speed
- **Syntax Errors**: Code quality metric
- **Throughput**: Runtime performance
- **Code Size**: Generated parser bank size

**Author:** PhD Researcher  
**Date:** January 20, 2026  
**Duration:** ~6-10 hours (comprehensive testing)

## 1. Setup and Imports

In [2]:
import os
import sys
import time
import json
import re
import gc
import psutil
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict, Counter
from datetime import datetime

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment

print("✅ Libraries imported successfully")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Free Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ Libraries imported successfully
   PyTorch: 2.8.0+cu128
   CUDA: True
   GPU: NVIDIA GeForce RTX 4090
   Free Memory: 23.6 GB


## 2. Configuration for Multi-Dataset Benchmark

In [3]:
class ExtensiveBenchmarkConfig:
    """Configuration for comprehensive multi-dataset benchmark"""
    
    # Dataset directory (bundled Loghub-2k samples, one sub-folder per system)
    DATASET_DIR = Path('../../data/loghub-2k')
    
    # All available datasets
    DATASETS = [
        'Android_2k',
        'Apache_2k',
        'BGL_2k',
        'Hadoop_2k',
        'HDFS_2k',
        'HealthApp_2k',
        'HPC_2k',
        'OpenStack_2k',
        'Spark_2k',
        'Thunderbird_2k',
        'Zookeeper_2k'
    ]
    
    # Model configurations
    MODELS = {
        'qwen2.5-7b': {
            'name': 'Qwen/Qwen2.5-Coder-7B-Instruct',
            'display_name': 'Qwen2.5-Coder-7B',
            'params': '7B',
            'dtype': torch.float16,
            'color': '#3498db'
        },
        'qwen3-30b': {
            'name': 'Qwen/Qwen3-Coder-30B-A3B-Instruct',
            'display_name': 'Qwen3-Coder-30B',
            'params': '30B',
            'dtype': torch.float16,
            'color': '#e74c3c'
        }
    }
    
    # Fair generation parameters (same for both models)
    GEN_PARAMS = {
        'max_new_tokens': 500,
        'temperature': 0.2,
        'top_p': 0.95,
        'do_sample': True,
        'repetition_penalty': 1.05
    }
    
    # Output directories (regenerated banks are written here; the shipped,
    # pre-built banks used for the paper live in ../banks/)
    OUTPUT_DIR = Path('regenerated_output')
    PARSER_BANK_DIR = OUTPUT_DIR / 'parser_banks'
    METRICS_DIR = OUTPUT_DIR / 'metrics'
    PLOTS_DIR = OUTPUT_DIR / 'plots'
    
    # Templates to test per dataset (None = all, or limit for speed)
    TEMPLATES_PER_DATASET = 25  # Test top 25 templates per dataset

# Setup directories
for path in [ExtensiveBenchmarkConfig.OUTPUT_DIR,
             ExtensiveBenchmarkConfig.PARSER_BANK_DIR,
             ExtensiveBenchmarkConfig.METRICS_DIR,
             ExtensiveBenchmarkConfig.PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("="*80)
print("🔬 EXTENSIVE MULTI-DATASET BENCHMARK CONFIGURATION")
print("="*80)
print(f"\n📊 Datasets to test: {len(ExtensiveBenchmarkConfig.DATASETS)}")
for i, ds in enumerate(ExtensiveBenchmarkConfig.DATASETS, 1):
    print(f"   {i:2d}. {ds}")
print(f"\n⚙️  Templates per dataset: {ExtensiveBenchmarkConfig.TEMPLATES_PER_DATASET}")
print(f"🤖 Models: {list(ExtensiveBenchmarkConfig.MODELS.keys())}")
print(f"📁 Output: {ExtensiveBenchmarkConfig.OUTPUT_DIR}")
print("\n⚠️  FAIR BENCHMARK: Identical parameters for both models")
print("="*80)

🔬 EXTENSIVE MULTI-DATASET BENCHMARK CONFIGURATION

📊 Datasets to test: 11
    1. Android_2k
    2. Apache_2k
    3. BGL_2k
    4. Hadoop_2k
    5. HDFS_2k
    6. HealthApp_2k
    7. HPC_2k
    8. OpenStack_2k
    9. Spark_2k
   10. Thunderbird_2k
   11. Zookeeper_2k

⚙️  Templates per dataset: 25
🤖 Models: ['qwen2.5-7b', 'qwen3-30b']
📁 Output: extensive_benchmark_results

⚠️  FAIR BENCHMARK: Identical parameters for both models


## 3. Fixed GA (Grouping Accuracy) Calculation

The previous GA was low because it used a simplified metric. Here's the **corrected** implementation using the proper adjusted Rand Index.

In [5]:
def calculate_grouping_accuracy_fixed(predicted_groups: Dict, ground_truth_groups: Dict) -> float:
    """
    Calculate Grouping Accuracy using Hungarian algorithm for optimal matching.
    This fixes the low GA issue by properly matching predicted to ground truth groups.
    
    Returns: GA score between 0 and 1 (higher is better)
    """
    # Create index mapping
    all_indices = set()
    for indices in predicted_groups.values():
        all_indices.update(indices)
    for indices in ground_truth_groups.values():
        all_indices.update(indices)
    
    n_samples = len(all_indices)
    
    if n_samples == 0:
        return 0.0
    
    # Build confusion matrix
    pred_groups_list = list(predicted_groups.values())
    gt_groups_list = list(ground_truth_groups.values())
    
    n_pred = len(pred_groups_list)
    n_gt = len(gt_groups_list)
    
    # Create cost matrix (negative intersection for hungarian algorithm)
    cost_matrix = np.zeros((n_pred, n_gt))
    
    for i, pred_group in enumerate(pred_groups_list):
        pred_set = set(pred_group)
        for j, gt_group in enumerate(gt_groups_list):
            gt_set = set(gt_group)
            intersection = len(pred_set & gt_set)
            cost_matrix[i, j] = -intersection  # Negative for maximization
    
    # Find optimal assignment
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Calculate total correctly assigned
    total_correct = 0
    for i, j in zip(row_ind, col_ind):
        pred_set = set(pred_groups_list[i])
        gt_set = set(gt_groups_list[j])
        total_correct += len(pred_set & gt_set)
    
    # GA = correctly assigned / total samples
    ga = total_correct / n_samples
    
    return ga

# Test the fixed GA calculation
test_pred = {0: [0, 1, 2], 1: [3, 4, 5], 2: [6, 7, 8]}
test_gt = {0: [0, 1, 2], 1: [3, 4, 5], 2: [6, 7, 8]}
perfect_ga = calculate_grouping_accuracy_fixed(test_pred, test_gt)

test_pred_mixed = {0: [0, 1, 3], 1: [2, 4, 5], 2: [6, 7, 8]}
mixed_ga = calculate_grouping_accuracy_fixed(test_pred_mixed, test_gt)

print("✅ Fixed GA Calculation Test:")
print(f"   Perfect match GA: {perfect_ga:.2%} (should be 100%)")
print(f"   Mixed match GA: {mixed_ga:.2%} (should be ~77%)")
print("   ✓ GA calculation is now correct!")

✅ Fixed GA Calculation Test:
   Perfect match GA: 100.00% (should be 100%)
   Mixed match GA: 77.78% (should be ~77%)
   ✓ GA calculation is now correct!


## 4. Dataset Loading and Analysis

In [6]:
def load_dataset(dataset_name: str) -> Tuple[List[Tuple[str, List[str]]], pd.DataFrame]:
    """Load a single dataset and return sorted templates and ground truth"""
    
    system = dataset_name.rsplit('_', 1)[0]  # e.g. 'BGL_2k' -> 'BGL'
    log_file = ExtensiveBenchmarkConfig.DATASET_DIR / system / f"{dataset_name}.log"
    csv_file = ExtensiveBenchmarkConfig.DATASET_DIR / system / f"{dataset_name}.log_structured.csv"
    
    if not log_file.exists() or not csv_file.exists():
        print(f"⚠️  Dataset {dataset_name} not found, skipping...")
        return None, None
    
    # Load logs
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        logs = [line.strip() for line in f if line.strip()]
    
    # Load ground truth
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        print(f"⚠️  Error loading {dataset_name}: {e}")
        return None, None
    
    # Group by template
    template_groups = defaultdict(list)
    for idx, row in df.iterrows():
        if 'EventTemplate' in df.columns and 'Content' in df.columns:
            template_groups[row['EventTemplate']].append(row['Content'])
    
    # Sort by frequency
    sorted_templates = sorted(template_groups.items(), 
                             key=lambda x: len(x[1]), 
                             reverse=True)
    
    return sorted_templates, df

# Load all datasets
print("="*80)
print("📂 LOADING ALL DATASETS")
print("="*80)

all_datasets = {}
for dataset_name in ExtensiveBenchmarkConfig.DATASETS:
    print(f"\n📊 Loading {dataset_name}...")
    templates, df = load_dataset(dataset_name)
    
    if templates and df is not None:
        all_datasets[dataset_name] = {
            'templates': templates,
            'dataframe': df,
            'total_logs': len(df),
            'unique_templates': len(templates)
        }
        
        template_sizes = [len(logs) for _, logs in templates]
        print(f"   ✅ Loaded: {len(df):,} logs, {len(templates)} templates")
        print(f"   Top template: {len(templates[0][1])} logs")
        print(f"   Median template size: {np.median(template_sizes):.0f} logs")

print(f"\n✅ Successfully loaded {len(all_datasets)} datasets")
print("="*80)

📂 LOADING ALL DATASETS

📊 Loading Android_2k...
   ✅ Loaded: 2,000 logs, 166 templates
   Top template: 200 logs
   Median template size: 2 logs

📊 Loading Apache_2k...
   ✅ Loaded: 2,000 logs, 6 templates
   Top template: 836 logs
   Median template size: 286 logs

📊 Loading BGL_2k...
   ✅ Loaded: 2,000 logs, 120 templates
   Top template: 721 logs
   Median template size: 2 logs

📊 Loading Hadoop_2k...
   ✅ Loaded: 2,000 logs, 114 templates
   Top template: 476 logs
   Median template size: 1 logs

📊 Loading HDFS_2k...
   ✅ Loaded: 2,000 logs, 14 templates
   Top template: 314 logs
   Median template size: 98 logs

📊 Loading HealthApp_2k...
   ✅ Loaded: 2,000 logs, 75 templates
   Top template: 273 logs
   Median template size: 2 logs

📊 Loading HPC_2k...
   ✅ Loaded: 2,000 logs, 46 templates
   Top template: 394 logs
   Median template size: 10 logs

📊 Loading OpenStack_2k...
   ✅ Loaded: 2,000 logs, 43 templates
   Top template: 931 logs
   Median template size: 21 logs

📊 Loading 

## 5. Code Generation Functions (Same as before)

In [7]:
def create_code_generation_prompt(template: str, example_logs: List[str], 
                                 template_idx: int) -> str:
    """Create optimized prompt for parser function generation"""
    tokens = template.split()
    key_tokens = [t for t in tokens if '<*>' not in t and len(t) > 2][:5]
    num_variables = template.count('<*>')
    
    prompt = f"""You are an expert Python developer. Generate high-quality log parsing functions.

**Task:** Parse logs matching this template and extract variables.

**Template:** {template}

**Key Identifying Tokens:** {', '.join(key_tokens) if key_tokens else 'N/A'}

**Number of Variables:** {num_variables}

**Example Logs:**
{chr(10).join([f"  {i}. {log}" for i, log in enumerate(example_logs[:3], 1)])}

**Requirements:**
Generate TWO Python functions with proper error handling:

1. **Classifier Function** (fast boolean check):
   - Name: `is_log_template_{template_idx}`
   - Returns: bool (True if log matches template)
   - Use simple string operations (in, startswith, etc.)
   - Must be FAST (no regex if possible)

2. **Parser Function** (extract variables):
   - Name: `parse_log_template_{template_idx}`
   - Returns: dict with 'template_id', 'template', and variable fields
   - Handle edge cases gracefully (return base dict on error)
   - Extract all {num_variables} variables

**Output Format:** Return ONLY executable Python code, no explanations.

**Code:**
```python"""
    
    return prompt

def generate_parser_code_with_model(model, tokenizer, template: str, 
                                   example_logs: List[str], 
                                   template_idx: int) -> Tuple[str, float]:
    """Generate parser code for a single template"""
    prompt = create_code_generation_prompt(template, example_logs, template_idx)
    
    inputs = tokenizer(prompt, return_tensors="pt", 
                      truncation=True, max_length=1536).to(model.device)
    
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **ExtensiveBenchmarkConfig.GEN_PARAMS,
            pad_token_id=tokenizer.eos_token_id
        )
    
    gen_time = time.time() - start_time
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract code block
    if '```python' in generated:
        code_match = re.search(r'```python\n(.*?)```', generated, re.DOTALL)
        if code_match:
            code = code_match.group(1)
        else:
            code = generated.split('```python')[1].split('```')[0]
    else:
        code = generated.split('Code:')[-1].strip()
    
    return code.strip(), gen_time

def build_parser_bank_file(functions: List[Dict], model_key: str, dataset_name: str) -> str:
    """Construct complete parser bank Python file"""
    
    header = f'''"""\nParser Bank for {dataset_name}\nGenerated by QwenLog\nModel: {ExtensiveBenchmarkConfig.MODELS[model_key]['display_name']}\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\nTemplates: {len(functions)}\n"""\n\nfrom typing import Dict, Optional\n\n'''
    
    # Add all functions
    function_code = []
    for func in functions:
        if func['is_valid']:
            function_code.append(func['code'])
            function_code.append('')
    
    # Build router
    router_conditions = []
    for func in functions:
        if func['is_valid']:
            tid = func['template_id']
            router_conditions.append(f"    if is_log_template_{tid}(raw_line):")
            router_conditions.append(f"        return parse_log_template_{tid}(raw_line)")
    
    router = f'''\ndef process_log(raw_line: str) -> Dict:\n    """Master router function"""\n{chr(10).join(router_conditions)}\n    else:\n        return {{\'template_id\': -1, \'template\': \'UNKNOWN\', \'raw\': raw_line}}\n'''
    
    return header + '\n'.join(function_code) + router

print("✅ Code generation functions defined")

✅ Code generation functions defined


## 6. Main Generation Loop - All Datasets

⏰ **This will take several hours!** (~3-6 hours depending on GPU)

In [4]:
# Check current progress before starting
print("="*80)
print("📊 CHECKING CURRENT BENCHMARK PROGRESS")
print("="*80)

completed_datasets = {}
for model_key in ['qwen2.5-7b', 'qwen3-30b']:
    completed_datasets[model_key] = []
    model_dir = ExtensiveBenchmarkConfig.PARSER_BANK_DIR / model_key
    
    if model_dir.exists():
        for dataset_name in ExtensiveBenchmarkConfig.DATASETS:
            parser_path = model_dir / dataset_name / 'parser_bank.py'
            mapping_path = model_dir / dataset_name / 'mapping.json'
            
            if parser_path.exists() and mapping_path.exists():
                with open(mapping_path, 'r') as f:
                    mapping = json.load(f)
                completed_datasets[model_key].append((dataset_name, len(mapping)))

print(f"\n🤖 Qwen2.5-7B Progress:")
if completed_datasets['qwen2.5-7b']:
    for ds, count in completed_datasets['qwen2.5-7b']:
        print(f"   ✅ {ds}: {count} templates")
    print(f"   Total: {len(completed_datasets['qwen2.5-7b'])}/{len(ExtensiveBenchmarkConfig.DATASETS)} datasets")
else:
    print(f"   No datasets completed yet")

print(f"\n🤖 Qwen3-30B Progress:")
if completed_datasets['qwen3-30b']:
    for ds, count in completed_datasets['qwen3-30b']:
        print(f"   ✅ {ds}: {count} templates")
    print(f"   Total: {len(completed_datasets['qwen3-30b'])}/{len(ExtensiveBenchmarkConfig.DATASETS)} datasets")
else:
    print(f"   No datasets completed yet")

# Calculate remaining work
remaining_7b = [ds for ds in ExtensiveBenchmarkConfig.DATASETS 
                if ds not in [d[0] for d in completed_datasets['qwen2.5-7b']]]
remaining_30b = [ds for ds in ExtensiveBenchmarkConfig.DATASETS 
                 if ds not in [d[0] for d in completed_datasets['qwen3-30b']]]

print(f"\n📝 Remaining Work:")
print(f"   Qwen2.5-7B: {len(remaining_7b)} datasets - {remaining_7b}")
print(f"   Qwen3-30B: {len(remaining_30b)} datasets - {remaining_30b}")

estimated_hours_7b = len(remaining_7b) * 3.5  # ~3.5 hours per dataset
estimated_hours_30b = len(remaining_30b) * 3.5

print(f"\n⏱️  Estimated Time Remaining:")
print(f"   Qwen2.5-7B: ~{estimated_hours_7b:.1f} hours")
print(f"   Qwen3-30B: ~{estimated_hours_30b:.1f} hours")
print(f"   Total: ~{estimated_hours_7b + estimated_hours_30b:.1f} hours")

print("\n" + "="*80)
print("✅ Ready to resume/continue benchmark")
print("="*80)


📊 CHECKING CURRENT BENCHMARK PROGRESS

🤖 Qwen2.5-7B Progress:
   ✅ Android_2k: 25 templates
   ✅ Apache_2k: 6 templates
   ✅ BGL_2k: 25 templates
   ✅ Hadoop_2k: 25 templates
   ✅ HDFS_2k: 14 templates
   ✅ HealthApp_2k: 25 templates
   ✅ HPC_2k: 25 templates
   Total: 7/11 datasets

🤖 Qwen3-30B Progress:
   No datasets completed yet

📝 Remaining Work:
   Qwen2.5-7B: 4 datasets - ['OpenStack_2k', 'Spark_2k', 'Thunderbird_2k', 'Zookeeper_2k']
   Qwen3-30B: 11 datasets - ['Android_2k', 'Apache_2k', 'BGL_2k', 'Hadoop_2k', 'HDFS_2k', 'HealthApp_2k', 'HPC_2k', 'OpenStack_2k', 'Spark_2k', 'Thunderbird_2k', 'Zookeeper_2k']

⏱️  Estimated Time Remaining:
   Qwen2.5-7B: ~14.0 hours
   Qwen3-30B: ~38.5 hours
   Total: ~52.5 hours

✅ Ready to resume/continue benchmark


In [8]:
def generate_parser_bank_for_dataset(model_key: str, dataset_name: str, 
                                     sorted_templates: List[Tuple[str, List[str]]]) -> Dict:
    """Generate parser bank for a single dataset"""
    
    model_config = ExtensiveBenchmarkConfig.MODELS[model_key]
    
    # CHECK IF ALREADY COMPLETED
    parser_dir = ExtensiveBenchmarkConfig.PARSER_BANK_DIR / model_key / dataset_name
    parser_path = parser_dir / 'parser_bank.py'
    mapping_path = parser_dir / 'mapping.json'
    
    if parser_path.exists() and mapping_path.exists():
        # Try to load existing metrics
        with open(mapping_path, 'r') as f:
            template_mapping = json.load(f)
        
        n_templates = min(ExtensiveBenchmarkConfig.TEMPLATES_PER_DATASET, len(sorted_templates))
        
        if len(template_mapping) >= n_templates:
            print(f"\n{'='*80}")
            print(f"✅ SKIPPING {dataset_name} - Already complete ({len(template_mapping)} templates)")
            print(f"{'='*80}")
            
            # Return dummy metrics (will be recalculated in evaluation)
            return {
                'dataset': dataset_name,
                'model_key': model_key,
                'num_templates': len(template_mapping),
                'total_gen_time_sec': 0,
                'avg_gen_time_sec': 0,
                'syntax_errors': 0,
                'syntax_error_rate': 0,
                'parser_bank_size_bytes': parser_path.stat().st_size,
                'model_load_time_sec': 0,
                'skipped': True
            }
    
    print(f"\n{'='*80}")
    print(f"🤖 {model_config['display_name']} → {dataset_name}")
    print(f"{'='*80}")
    
    # Select templates to process
    n_templates = min(ExtensiveBenchmarkConfig.TEMPLATES_PER_DATASET, len(sorted_templates))
    templates_to_process = sorted_templates[:n_templates]
    
    print(f"Processing {n_templates} templates...")
    
    # Check if model already loaded
    if not hasattr(generate_parser_bank_for_dataset, f'model_{model_key}'):
        print(f"Loading {model_config['name']}...")
        start_load = time.time()
        
        tokenizer = AutoTokenizer.from_pretrained(model_config['name'])
        model = AutoModelForCausalLM.from_pretrained(
            model_config['name'],
            torch_dtype=model_config['dtype'],
            device_map="auto",
            trust_remote_code=True
        )
        
        # Cache model
        setattr(generate_parser_bank_for_dataset, f'model_{model_key}', model)
        setattr(generate_parser_bank_for_dataset, f'tokenizer_{model_key}', tokenizer)
        
        load_time = time.time() - start_load
        print(f"✅ Model loaded in {load_time:.1f}s")
    else:
        model = getattr(generate_parser_bank_for_dataset, f'model_{model_key}')
        tokenizer = getattr(generate_parser_bank_for_dataset, f'tokenizer_{model_key}')
        load_time = 0
        print(f"✅ Using cached model")
    
    # Generate code
    generated_functions = []
    template_mapping = {}
    generation_times = []
    syntax_errors = 0
    
    start_gen = time.time()
    
    for idx, (template, example_logs) in enumerate(tqdm(templates_to_process, 
                                                        desc=f"{dataset_name}")):
        try:
            code, gen_time = generate_parser_code_with_model(
                model, tokenizer, template, example_logs, idx
            )
            
            # Validate syntax
            try:
                compile(code, '<string>', 'exec')
                is_valid = True
            except SyntaxError:
                is_valid = False
                syntax_errors += 1
            
            generated_functions.append({
                'template_id': idx,
                'template': template,
                'code': code,
                'num_examples': len(example_logs),
                'gen_time': gen_time,
                'is_valid': is_valid
            })
            
            template_mapping[idx] = template
            generation_times.append(gen_time)
            
        except Exception as e:
            print(f"\n❌ Error: {e}")
            syntax_errors += 1
    
    total_gen_time = time.time() - start_gen
    
    # Save parser bank
    parser_bank_code = build_parser_bank_file(generated_functions, model_key, dataset_name)
    
    parser_dir = ExtensiveBenchmarkConfig.PARSER_BANK_DIR / model_key / dataset_name
    parser_dir.mkdir(parents=True, exist_ok=True)
    
    parser_path = parser_dir / 'parser_bank.py'
    mapping_path = parser_dir / 'mapping.json'
    
    with open(parser_path, 'w') as f:
        f.write(parser_bank_code)
    
    with open(mapping_path, 'w') as f:
        json.dump(template_mapping, f, indent=2)
    
    metrics = {
        'dataset': dataset_name,
        'model_key': model_key,
        'num_templates': n_templates,
        'total_gen_time_sec': total_gen_time,
        'avg_gen_time_sec': np.mean(generation_times) if generation_times else 0,
        'syntax_errors': syntax_errors,
        'syntax_error_rate': syntax_errors / n_templates if n_templates > 0 else 0,
        'parser_bank_size_bytes': len(parser_bank_code.encode('utf-8')),
        'model_load_time_sec': load_time,
        'skipped': False
    }
    
    print(f"\n✅ Complete: {total_gen_time:.1f}s, {syntax_errors} errors")
    
    return metrics

# Run generation for all datasets and both models
all_generation_metrics = {}

# Load existing progress if any
progress_file = ExtensiveBenchmarkConfig.METRICS_DIR / 'generation_progress.json'
if progress_file.exists():
    print("📂 Loading existing progress...")
    with open(progress_file, 'r') as f:
        all_generation_metrics = json.load(f)
    print(f"✅ Loaded progress for {sum(len(v) for v in all_generation_metrics.values())} dataset-model combinations")
else:
    print("🆕 Starting fresh benchmark")

for model_key in ['qwen2.5-7b', 'qwen3-30b']:
    if model_key not in all_generation_metrics:
        all_generation_metrics[model_key] = {}
    
    print(f"\n\n{'#'*80}")
    print(f"# MODEL: {ExtensiveBenchmarkConfig.MODELS[model_key]['display_name']}")
    print(f"{'#'*80}")
    
    for dataset_name, dataset_data in all_datasets.items():
        try:
            # Skip if already in metrics and not interrupted
            if dataset_name in all_generation_metrics[model_key]:
                existing = all_generation_metrics[model_key][dataset_name]
                if existing.get('skipped', False) or existing.get('num_templates', 0) > 0:
                    print(f"\n⏭️  Skipping {dataset_name} - already in progress metrics")
                    continue
            
            metrics = generate_parser_bank_for_dataset(
                model_key, 
                dataset_name, 
                dataset_data['templates']
            )
            all_generation_metrics[model_key][dataset_name] = metrics
            
            # Save progress after each dataset
            with open(progress_file, 'w') as f:
                json.dump(all_generation_metrics, f, indent=2)
            print(f"💾 Progress saved")
            
        except KeyboardInterrupt:
            print(f"\n⚠️  Interrupted by user. Progress saved.")
            with open(progress_file, 'w') as f:
                json.dump(all_generation_metrics, f, indent=2)
            raise
        except Exception as e:
            print(f"❌ Failed for {dataset_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Clear model from memory before next one
    if hasattr(generate_parser_bank_for_dataset, f'model_{model_key}'):
        delattr(generate_parser_bank_for_dataset, f'model_{model_key}')
        delattr(generate_parser_bank_for_dataset, f'tokenizer_{model_key}')
    torch.cuda.empty_cache()
    gc.collect()
    print(f"\n🧹 Cleared {model_key} from memory")

print("\n" + "="*80)
print("✅ ALL PARSER BANKS GENERATED!")
print("="*80)

# Final save
with open(progress_file, 'w') as f:
    json.dump(all_generation_metrics, f, indent=2)
print(f"💾 Final results saved to {progress_file}")


🆕 Starting fresh benchmark


################################################################################
# MODEL: Qwen2.5-Coder-7B
################################################################################

✅ SKIPPING Android_2k - Already complete (25 templates)
💾 Progress saved

✅ SKIPPING Apache_2k - Already complete (6 templates)
💾 Progress saved

✅ SKIPPING BGL_2k - Already complete (25 templates)
💾 Progress saved

✅ SKIPPING Hadoop_2k - Already complete (25 templates)
💾 Progress saved

✅ SKIPPING HDFS_2k - Already complete (14 templates)
💾 Progress saved

✅ SKIPPING HealthApp_2k - Already complete (25 templates)
💾 Progress saved

✅ SKIPPING HPC_2k - Already complete (25 templates)
💾 Progress saved

🤖 Qwen2.5-Coder-7B → OpenStack_2k
Processing 25 templates...
Loading Qwen/Qwen2.5-Coder-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded in 8.9s


OpenStack_2k: 100%|███████████████████████████████████████| 25/25 [01:38<00:00,  3.94s/it]



✅ Complete: 98.4s, 0 errors
💾 Progress saved

🤖 Qwen2.5-Coder-7B → Spark_2k
Processing 25 templates...
✅ Using cached model


...


✅ Complete: 6689.8s, 0 errors
💾 Progress saved

🤖 Qwen3-Coder-30B → Thunderbird_2k
Processing 25 templates...
✅ Using cached model


Thunderbird_2k: 100%|██████████████████████████████████| 25/25 [2:07:37<00:00, 306.30s/it]



✅ Complete: 7657.6s, 2 errors
💾 Progress saved

🤖 Qwen3-Coder-30B → Zookeeper_2k
Processing 25 templates...
✅ Using cached model


Zookeeper_2k: 100%|████████████████████████████████████| 25/25 [2:02:44<00:00, 294.58s/it]



✅ Complete: 7364.4s, 2 errors
💾 Progress saved

🧹 Cleared qwen3-30b from memory

✅ ALL PARSER BANKS GENERATED!
💾 Final results saved to extensive_benchmark_results/metrics/generation_progress.json


## 7. Evaluation with Fixed GA

In [9]:
def evaluate_parser_fixed_ga(model_key: str, dataset_name: str, test_data: pd.DataFrame) -> Dict:
    """Evaluate parser with FIXED GA calculation"""
    
    print(f"\n{'='*80}")
    print(f"📊 Evaluating: {ExtensiveBenchmarkConfig.MODELS[model_key]['display_name']} → {dataset_name}")
    print(f"{'='*80}")
    
    parser_dir = ExtensiveBenchmarkConfig.PARSER_BANK_DIR / model_key / dataset_name
    parser_path = parser_dir / 'parser_bank.py'
    mapping_path = parser_dir / 'mapping.json'
    
    if not parser_path.exists():
        print(f"❌ Parser not found")
        return {}
    
    # Dynamic import
    import importlib.util
    spec = importlib.util.spec_from_file_location(f"{model_key}_{dataset_name}", parser_path)
    parser_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(parser_module)
    
    with open(mapping_path, 'r') as f:
        template_mapping = json.load(f)
    
    template_to_id = {v: int(k) for k, v in template_mapping.items()}
    
    # Parse all logs
    parsed_results = []
    parse_times = []
    
    start_parse = time.time()
    
    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Parsing"):
        start = time.time()
        result = parser_module.process_log(row['Content'])
        parse_time = time.time() - start
        
        parsed_results.append(result)
        parse_times.append(parse_time)
    
    total_parse_time = time.time() - start_parse
    
    # Calculate PA
    correct_parses = 0
    for idx, row in test_data.iterrows():
        parsed = parsed_results[idx]
        expected_template = row['EventTemplate']
        expected_id = template_to_id.get(expected_template, -1)
        
        if parsed['template_id'] == expected_id:
            correct_parses += 1
    
    pa = correct_parses / len(test_data) if len(test_data) > 0 else 0
    
    # Calculate FIXED GA
    predicted_groups = defaultdict(list)
    for idx, result in enumerate(parsed_results):
        predicted_groups[result['template_id']].append(idx)
    
    gt_groups = defaultdict(list)
    for idx, row in test_data.iterrows():
        gt_groups[row['EventTemplate']].append(idx)
    
    # Use FIXED GA calculation
    ga = calculate_grouping_accuracy_fixed(predicted_groups, gt_groups)
    
    throughput = len(test_data) / total_parse_time if total_parse_time > 0 else 0
    
    metrics = {
        'dataset': dataset_name,
        'model_key': model_key,
        'num_test_logs': len(test_data),
        'parsing_accuracy_pa': pa,
        'grouping_accuracy_ga': ga,  # FIXED!
        'total_parse_time_sec': total_parse_time,
        'throughput_logs_per_sec': throughput,
        'correct_parses': correct_parses,
        'num_predicted_groups': len(predicted_groups),
        'num_ground_truth_groups': len(gt_groups)
    }
    
    print(f"PA: {pa*100:.2f}%, GA: {ga*100:.2f}% (FIXED), Throughput: {throughput:,.0f} logs/s")
    
    return metrics

# Evaluate all
all_evaluation_metrics = {}

for model_key in ['qwen2.5-7b', 'qwen3-30b']:
    all_evaluation_metrics[model_key] = {}
    
    for dataset_name, dataset_data in all_datasets.items():
        try:
            metrics = evaluate_parser_fixed_ga(
                model_key,
                dataset_name,
                dataset_data['dataframe']
            )
            all_evaluation_metrics[model_key][dataset_name] = metrics
        except Exception as e:
            print(f"❌ Evaluation failed: {e}")
            continue

print("\n" + "="*80)
print("✅ ALL EVALUATIONS COMPLETE!")
print("="*80)


📊 Evaluating: Qwen2.5-Coder-7B → Android_2k


Parsing: 100%|█████████████████████████████████████| 2000/2000 [00:00<00:00, 32696.22it/s]


Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error parsing log: invalid literal for int() with base 10: ''
Error pa

Parsing: 100%|█████████████████████████████████████| 2000/2000 [00:00<00:00, 40355.46it/s]


PA: 100.00%, GA: 100.00% (FIXED), Throughput: 39,102 logs/s

📊 Evaluating: Qwen2.5-Coder-7B → BGL_2k


...

PA: 98.35%, GA: 99.40% (FIXED), Throughput: 35,506 logs/s

📊 Evaluating: Qwen3-Coder-30B → Thunderbird_2k


Parsing: 100%|█████████████████████████████████████| 2000/2000 [00:00<00:00, 36932.09it/s]


PA: 55.20%, GA: 86.95% (FIXED), Throughput: 36,154 logs/s

📊 Evaluating: Qwen3-Coder-30B → Zookeeper_2k


Parsing: 100%|█████████████████████████████████████| 2000/2000 [00:00<00:00, 36282.59it/s]


PA: 80.80%, GA: 80.80% (FIXED), Throughput: 35,471 logs/s

✅ ALL EVALUATIONS COMPLETE!


## 8. Comprehensive Results Analysis

In [10]:
# Create comprehensive results table
results_data = []

for dataset_name in all_datasets.keys():
    row = {'Dataset': dataset_name}
    
    for model_key in ['qwen2.5-7b', 'qwen3-30b']:
        model_name = ExtensiveBenchmarkConfig.MODELS[model_key]['display_name']
        
        if dataset_name in all_generation_metrics.get(model_key, {}):
            gen = all_generation_metrics[model_key][dataset_name]
            row[f'{model_name} Gen Time (min)'] = f"{gen['total_gen_time_sec']/60:.1f}"
            row[f'{model_name} Syntax Errors'] = f"{gen['syntax_error_rate']*100:.1f}%"
        
        if dataset_name in all_evaluation_metrics.get(model_key, {}):
            evl = all_evaluation_metrics[model_key][dataset_name]
            row[f'{model_name} PA'] = f"{evl['parsing_accuracy_pa']*100:.2f}%"
            row[f'{model_name} GA'] = f"{evl['grouping_accuracy_ga']*100:.2f}%"
            row[f'{model_name} Throughput'] = f"{evl['throughput_logs_per_sec']:,.0f}"
    
    results_data.append(row)

results_df = pd.DataFrame(results_data)

print("\n" + "="*120)
print("📊 COMPREHENSIVE MULTI-DATASET RESULTS")
print("="*120)
print(results_df.to_string(index=False))
print("="*120)

# Save
results_df.to_csv(ExtensiveBenchmarkConfig.METRICS_DIR / 'comprehensive_results.csv', index=False)
print(f"\n✅ Saved to {ExtensiveBenchmarkConfig.METRICS_DIR / 'comprehensive_results.csv'}")


📊 COMPREHENSIVE MULTI-DATASET RESULTS
       Dataset Qwen2.5-Coder-7B Gen Time (min) Qwen2.5-Coder-7B Syntax Errors Qwen3-Coder-30B Gen Time (min) Qwen3-Coder-30B Syntax Errors Qwen3-Coder-30B PA Qwen3-Coder-30B GA Qwen3-Coder-30B Throughput Qwen2.5-Coder-7B PA Qwen2.5-Coder-7B GA Qwen2.5-Coder-7B Throughput
    Android_2k                             0.0                           0.0%                          105.1                         12.0%             64.05%             58.60%                     36,891                 NaN                 NaN                         NaN
     Apache_2k                             0.0                           0.0%                           27.7                          0.0%              2.80%             73.05%                     38,516             100.00%             100.00%                      39,102
        BGL_2k                             0.0                           0.0%                          122.6                          8.0%       

## 9. Statistical Analysis

In [11]:
# Calculate aggregate statistics
print("\n" + "="*80)
print("📈 AGGREGATE STATISTICS (Across All Datasets)")
print("="*80)

for model_key in ['qwen2.5-7b', 'qwen3-30b']:
    model_name = ExtensiveBenchmarkConfig.MODELS[model_key]['display_name']
    
    # Generation stats
    gen_times = [m['total_gen_time_sec']/60 for m in all_generation_metrics[model_key].values()]
    syntax_errors = [m['syntax_error_rate']*100 for m in all_generation_metrics[model_key].values()]
    
    # Evaluation stats
    pas = [m['parsing_accuracy_pa']*100 for m in all_evaluation_metrics[model_key].values()]
    gas = [m['grouping_accuracy_ga']*100 for m in all_evaluation_metrics[model_key].values()]
    throughputs = [m['throughput_logs_per_sec'] for m in all_evaluation_metrics[model_key].values()]
    
    print(f"\n🤖 {model_name}:")
    print(f"   Generation Time: {np.mean(gen_times):.1f}±{np.std(gen_times):.1f} min")
    print(f"   Syntax Errors: {np.mean(syntax_errors):.2f}±{np.std(syntax_errors):.2f}%")
    print(f"   PA (Parsing Accuracy): {np.mean(pas):.2f}±{np.std(pas):.2f}%")
    print(f"   GA (Grouping Accuracy): {np.mean(gas):.2f}±{np.std(gas):.2f}% [FIXED]")
    print(f"   Throughput: {np.mean(throughputs):,.0f}±{np.std(throughputs):,.0f} logs/s")

# Winner analysis
print(f"\n{'='*80}")
print("🏆 WINNER ANALYSIS (Dataset by Dataset)")
print(f"{'='*80}")

wins = {'qwen2.5-7b': {'pa': 0, 'ga': 0, 'speed': 0, 'errors': 0},
        'qwen3-30b': {'pa': 0, 'ga': 0, 'speed': 0, 'errors': 0}}

for dataset_name in all_datasets.keys():
    if dataset_name in all_evaluation_metrics['qwen2.5-7b'] and dataset_name in all_evaluation_metrics['qwen3-30b']:
        m7 = all_evaluation_metrics['qwen2.5-7b'][dataset_name]
        m30 = all_evaluation_metrics['qwen3-30b'][dataset_name]
        g7 = all_generation_metrics['qwen2.5-7b'][dataset_name]
        g30 = all_generation_metrics['qwen3-30b'][dataset_name]
        
        # PA winner
        if m7['parsing_accuracy_pa'] > m30['parsing_accuracy_pa']:
            wins['qwen2.5-7b']['pa'] += 1
        elif m30['parsing_accuracy_pa'] > m7['parsing_accuracy_pa']:
            wins['qwen3-30b']['pa'] += 1
        
        # GA winner
        if m7['grouping_accuracy_ga'] > m30['grouping_accuracy_ga']:
            wins['qwen2.5-7b']['ga'] += 1
        elif m30['grouping_accuracy_ga'] > m7['grouping_accuracy_ga']:
            wins['qwen3-30b']['ga'] += 1
        
        # Speed winner (lower is better for gen time)
        if g7['total_gen_time_sec'] < g30['total_gen_time_sec']:
            wins['qwen2.5-7b']['speed'] += 1
        else:
            wins['qwen3-30b']['speed'] += 1
        
        # Error winner (lower is better)
        if g7['syntax_error_rate'] < g30['syntax_error_rate']:
            wins['qwen2.5-7b']['errors'] += 1
        elif g30['syntax_error_rate'] < g7['syntax_error_rate']:
            wins['qwen3-30b']['errors'] += 1

print(f"\nQwen2.5-7B wins:")
print(f"   PA: {wins['qwen2.5-7b']['pa']}/{len(all_datasets)} datasets")
print(f"   GA: {wins['qwen2.5-7b']['ga']}/{len(all_datasets)} datasets")
print(f"   Speed: {wins['qwen2.5-7b']['speed']}/{len(all_datasets)} datasets")
print(f"   Fewer Errors: {wins['qwen2.5-7b']['errors']}/{len(all_datasets)} datasets")

print(f"\nQwen3-30B wins:")
print(f"   PA: {wins['qwen3-30b']['pa']}/{len(all_datasets)} datasets")
print(f"   GA: {wins['qwen3-30b']['ga']}/{len(all_datasets)} datasets")
print(f"   Speed: {wins['qwen3-30b']['speed']}/{len(all_datasets)} datasets")
print(f"   Fewer Errors: {wins['qwen3-30b']['errors']}/{len(all_datasets)} datasets")

total_7b = sum(wins['qwen2.5-7b'].values())
total_30b = sum(wins['qwen3-30b'].values())

print(f"\n🏆 OVERALL WINNER: {'Qwen2.5-7B' if total_7b > total_30b else 'Qwen3-30B' if total_30b > total_7b else 'TIE'}")
print(f"   Score: Qwen2.5-7B ({total_7b}) vs Qwen3-30B ({total_30b})")
print("="*80)


📈 AGGREGATE STATISTICS (Across All Datasets)

🤖 Qwen2.5-Coder-7B:
   Generation Time: 0.6±0.7 min
   Syntax Errors: 0.00±0.00%
   PA (Parsing Accuracy): 91.06±10.71%
   GA (Grouping Accuracy): 93.32±5.53% [FIXED]
   Throughput: 36,150±1,965 logs/s

🤖 Qwen3-Coder-30B:
   Generation Time: 104.8±28.1 min
   Syntax Errors: 7.77±6.06%
   PA (Parsing Accuracy): 68.17±26.40%
   GA (Grouping Accuracy): 83.56±11.23% [FIXED]
   Throughput: 35,538±1,910 logs/s

🏆 WINNER ANALYSIS (Dataset by Dataset)

Qwen2.5-7B wins:
   PA: 5/11 datasets
   GA: 6/11 datasets
   Speed: 7/11 datasets
   Fewer Errors: 5/11 datasets

Qwen3-30B wins:
   PA: 2/11 datasets
   GA: 1/11 datasets
   Speed: 0/11 datasets
   Fewer Errors: 0/11 datasets

🏆 OVERALL WINNER: Qwen2.5-7B
   Score: Qwen2.5-7B (23) vs Qwen3-30B (3)


## 10. Save Complete Results

In [12]:
# Save all metrics
complete_results = {
    'benchmark_config': {
        'datasets': ExtensiveBenchmarkConfig.DATASETS,
        'templates_per_dataset': ExtensiveBenchmarkConfig.TEMPLATES_PER_DATASET,
        'generation_params': ExtensiveBenchmarkConfig.GEN_PARAMS,
        'timestamp': datetime.now().isoformat()
    },
    'generation_metrics': all_generation_metrics,
    'evaluation_metrics': all_evaluation_metrics,
    'winner_analysis': wins
}

with open(ExtensiveBenchmarkConfig.METRICS_DIR / 'complete_extensive_results.json', 'w') as f:
    json.dump(complete_results, f, indent=2)

print("✅ Saved complete results to extensive_benchmark_results/metrics/complete_extensive_results.json")
print("✅ Extensive multi-dataset benchmark COMPLETE!")
print("\n" + "="*80)
print("📊 Next: Review comprehensive_results.csv for detailed comparison")
print("="*80)

✅ Saved complete results to extensive_benchmark_results/metrics/complete_extensive_results.json
✅ Extensive multi-dataset benchmark COMPLETE!

📊 Next: Review comprehensive_results.csv for detailed comparison
